# Dataset Deep Dive

This notebook demonstrates how to explore a specific dataset in detail using the Carbon Arc Data Library API.

We'll cover:
1. Getting dataset information
2. Exploring the data dictionary (schema)
3. Viewing sample data
4. Understanding bulk data options

## Setup

In [ ]:
import os
import json
from dotenv import load_dotenv
from carbonarc import CarbonArcClient
import pandas as pd

load_dotenv()

API_AUTH_TOKEN = os.getenv("API_AUTH_TOKEN")
ca = CarbonArcClient(API_AUTH_TOKEN)

print("Client initialized successfully!")

## 1. Select a Dataset

First, let's pick a dataset to explore. We'll use the Credit Card dataset as an example.

In [ ]:
# Choose a dataset to explore
DATASET_ID = "CA0056"  # Credit Card – US Complete Panel

# You can also explore other datasets:
# DATASET_ID = "CA0030"  # Clickstream
# DATASET_ID = "CA0054"  # App Intelligence
# DATASET_ID = "CA0049"  # Medical & Pharmacy Open Claims

## 2. Get Dataset Information

Retrieve comprehensive metadata about the dataset.

In [ ]:
# Fetch dataset information
info = ca.data.get_dataset_information(DATASET_ID)

print(f"Dataset: {info['dataset_name']}")
print(f"ID: {DATASET_ID}")
print(f"\nDescription:\n{info['description']}")

In [ ]:
# Extract key metadata
data = info.get('data', {})

metadata = {
    'Type': data.get('Type'),
    'Provider': info.get('provider_name'),
    'Frequency': data.get('Frequency'),
    'History': data.get('History'),
    'Lag': data.get('Lag'),
    'Panel Size': data.get('Panel Size'),
    'Granularity': data.get('Granularity'),
    'Bulk Data': data.get('Bulk Data'),
    'Bias': data.get('Bias'),
}

print("Key Metadata:")
print("=" * 50)
for key, value in metadata.items():
    if value:
        print(f"{key:15} {value}")

In [ ]:
# Coverage details
coverage = data.get('Coverage', {})
print("Coverage:")
print("=" * 50)
if isinstance(coverage, dict):
    for key, value in coverage.items():
        print(f"  {key}: {value}")
elif isinstance(coverage, list):
    for item in coverage:
        print(f"  - {item}")

In [ ]:
# Geographic availability
geo = data.get('Geographic Availability', [])
print(f"Geographic Availability: {', '.join(geo) if geo else 'N/A'}")

In [ ]:
# Key metrics
metrics = data.get('Key Metrics', [])
print("Key Metrics:")
print("=" * 50)
for metric in metrics:
    print(f"  - {metric}")

In [ ]:
# Example use cases
use_cases = data.get('Example Use Cases', [])
print("Example Use Cases:")
print("=" * 50)
for i, uc in enumerate(use_cases, 1):
    print(f"  {i}. {uc}")

## 3. Data Dictionary (Schema)

Get the field definitions, data types, and descriptions for all columns in the dataset.

In [ ]:
# Fetch the data dictionary
data_dict = ca.data.get_data_dictionary(DATASET_ID)

print(f"Data Dictionary for {DATASET_ID}")
print("=" * 60)

In [ ]:
# Display the raw structure
print(json.dumps(data_dict, indent=2)[:3000])  # First 3000 chars

In [ ]:
# If data dictionary is a list of fields, convert to DataFrame
if isinstance(data_dict, list):
    df_schema = pd.DataFrame(data_dict)
    display(df_schema)
elif isinstance(data_dict, dict) and 'fields' in data_dict:
    df_schema = pd.DataFrame(data_dict['fields'])
    display(df_schema)
else:
    print("Data dictionary structure:")
    print(type(data_dict))
    print(data_dict.keys() if isinstance(data_dict, dict) else data_dict)

## 4. Data Sample

Preview actual data rows from the dataset.

In [ ]:
# Fetch sample data
sample = ca.data.get_data_sample(DATASET_ID)

print(f"Sample Data for {DATASET_ID}")
print("=" * 60)

In [ ]:
# Display sample structure
print(json.dumps(sample, indent=2)[:3000])  # First 3000 chars

In [ ]:
# If sample contains rows, convert to DataFrame
if isinstance(sample, list):
    df_sample = pd.DataFrame(sample)
    display(df_sample.head(20))
elif isinstance(sample, dict) and 'data' in sample:
    df_sample = pd.DataFrame(sample['data'])
    display(df_sample.head(20))
elif isinstance(sample, dict) and 'rows' in sample:
    df_sample = pd.DataFrame(sample['rows'])
    display(df_sample.head(20))

## 5. Bulk Data Availability

Check if bulk data export is available and explore the manifest.

In [ ]:
# Check bulk data availability
bulk_available = data.get('Bulk Data') == 'Available'
print(f"Bulk Data Available: {bulk_available}")

if bulk_available:
    print("\nYou can download bulk data using:")
    print("  - ca.data.get_data_manifest(dataset_id, created_since)")
    print("  - ca.data.buy_data(dataset_id, file_urls)")
    print("  - ca.data.download_file(file_id, directory)")
    print("\nSee bulk_data_coverage.ipynb for coverage analysis examples.")

In [ ]:
# Preview manifest structure (without actually buying)
if bulk_available:
    from datetime import datetime, timedelta
    
    # Get manifest for files created in the last 30 days
    since_date = (datetime.now() - timedelta(days=30)).strftime('%Y-%m-%dT%H:%M:%S')
    
    try:
        manifest = ca.data.get_data_manifest(DATASET_ID, created_since=since_date)
        
        files = manifest.get('datasources', [])
        print(f"Files available (last 30 days): {len(files)}")
        
        if files:
            print("\nSample file info:")
            sample_file = files[0]
            for key, value in sample_file.items():
                if key != 'url':  # Don't print full URL
                    print(f"  {key}: {value}")
    except Exception as e:
        print(f"Could not fetch manifest: {e}")

## 6. Summary

Compile all the information into a summary report.

In [ ]:
print("=" * 60)
print(f"DATASET SUMMARY: {info['dataset_name']}")
print("=" * 60)
print(f"\nID: {DATASET_ID}")
print(f"Type: {data.get('Type')}")
print(f"Provider: {info.get('provider_name', 'N/A')}")
print(f"\nUpdate Frequency: {data.get('Frequency')}")
print(f"Data Lag: {data.get('Lag')}")
print(f"History: {data.get('History')}")
print(f"\nPanel Size: {data.get('Panel Size')}")
print(f"Granularity: {data.get('Granularity')}")
print(f"\nBulk Data: {data.get('Bulk Data')}")
print(f"Geographic Availability: {', '.join(geo) if geo else 'N/A'}")
print(f"\nKnown Bias: {data.get('Bias', 'None identified')}")
print("\n" + "=" * 60)